In [0]:
import os
import time
from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml.regression import GBTRegressor, RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator

import mlflow
import mlflow.spark
from mlflow.models.signature import infer_signature

CATALOG = "fahrlytics"
GOLD    = f"{CATALOG}.gold"
MODEL_NAME = f"{CATALOG}.models.demand_forecaster_gbt"

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.models")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.models.mlflow_staging")

# Configure UC Volume staging path for MLflow SparkML models
UC_VOLUME_TMP = f"/Volumes/{CATALOG}/models/mlflow_staging"
os.environ["MLFLOW_DFS_TMP"] = UC_VOLUME_TMP

mlflow.set_experiment(f"/Users/{spark.sql('SELECT current_user()').collect()[0][0]}/fahrlytics_demand_forecasting")

print(f"Catalog             : {CATALOG}")
print(f"Source Feature Table : {GOLD}.trip_features")
print(f"MLflow Temp Volume  : {UC_VOLUME_TMP}")
print(f"Target Model Name   : {MODEL_NAME}")

In [0]:
print("Loading Gold trip features dataset...")

raw_df = spark.table(f"{GOLD}.trip_features")

# Cast regression target to Double
df = (
    raw_df
    .withColumn("label", F.col("trip_count").cast("double"))
    .withColumn("is_weekend", F.col("is_weekend").cast("double"))
)

total_intervals = df.count()
date_min = df.select(F.min("trip_date")).collect()[0][0]
date_max = df.select(F.max("trip_date")).collect()[0][0]

print(f"Total City-Hourly Records : {total_intervals:,}")
print(f"Historical Temporal Span   : {date_min} to {date_max}")

In [0]:
# We train on historical data up to 2026-09-01 and test on September 2026.
SPLIT_DATE = "2026-09-01"

train_df = df.filter(F.col("trip_date") < SPLIT_DATE)
test_df  = df.filter(F.col("trip_date") >= SPLIT_DATE)

train_count = train_df.count()
test_count  = test_df.count()

print(f"Training Range (< {SPLIT_DATE})  : {train_count:,} hourly bins")
print(f"Testing Range  (>= {SPLIT_DATE}) : {test_count:,} hourly bins ({round(test_count / total_intervals * 100, 1)}% holdout)")

In [0]:
categorical_cols = ["city", "weather_condition", "day_type"]

numerical_cols = [
    # Calendar & Cyclical signals
    "pickup_hour",
    "day_of_week",
    "month",
    "is_weekend",
    "hour_sin",
    "hour_cos",
    "dow_sin",
    "dow_cos",
    # Operational & Dynamic Pricing Context
    "avg_surge_multiplier",
    "max_surge_multiplier",
    "avg_fare_eur",
    "cancellation_rate",
    "avg_distance_km",
    "avg_duration_min",
    # Fleet Mix Counts
    "fleet_standard_trips",
    "fleet_comfort_trips",
    "fleet_van_trips",
    # Temporal Lag Predictors
    "demand_lag_1h",
    "demand_lag_24h",
    "demand_lag_168h",
    "rolling_avg_demand_24h"
]

stages = []

# Index and One-Hot Encode Categorical Columns
for c in categorical_cols:
    indexer = StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep")
    encoder = OneHotEncoder(inputCols=[f"{c}_idx"], outputCols=[f"{c}_vec"])
    stages += [indexer, encoder]

# Assemble and Scale Feature Vectors
assembler_inputs = [f"{c}_vec" for c in categorical_cols] + numerical_cols
assembler = VectorAssembler(inputCols=assembler_inputs, outputCol="raw_features", handleInvalid="keep")
scaler    = StandardScaler(inputCol="raw_features", outputCol="features", withStd=True, withMean=False)

stages += [assembler, scaler]

# Gradient-Boosted Tree Regressor
gbt = GBTRegressor(
    featuresCol="features",
    labelCol="label",
    maxIter=40,
    maxDepth=6,
    stepSize=0.08,
    seed=42
)
stages.append(gbt)

pipeline = Pipeline(stages=stages)

In [0]:
evaluator_rmse = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="rmse")
evaluator_mae  = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="mae")
evaluator_r2   = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="r2")

print("Training GBT Demand Forecasting Model...")

with mlflow.start_run(run_name="gbt_demand_forecasting_hourly") as run:
    run_id = run.info.run_id
    t0 = time.time()
    
    # Fit pipeline
    model = pipeline.fit(train_df)
    train_duration = time.time() - t0
    
    # Predict on unseen future holdout (September 2026)
    predictions = model.transform(test_df)
    
    # Clip any negative predictions to zero (demand cannot be negative)
    predictions = predictions.withColumn(
        "prediction",
        F.when(F.col("prediction") < 0.0, 0.0).otherwise(F.round(F.col("prediction"), 1))
    )
    
    # Compute metrics
    rmse = evaluator_rmse.evaluate(predictions)
    mae  = evaluator_mae.evaluate(predictions)
    r2   = evaluator_r2.evaluate(predictions)
    
    # Log parameters
    mlflow.log_param("model_type", "GBTRegressor")
    mlflow.log_param("max_iter", 40)
    mlflow.log_param("max_depth", 6)
    mlflow.log_param("step_size", 0.08)
    mlflow.log_param("split_date", SPLIT_DATE)
    mlflow.log_param("train_samples", train_count)
    mlflow.log_param("test_samples", test_count)
    
    # Log evaluation metrics
    mlflow.log_metric("test_rmse", rmse)
    mlflow.log_metric("test_mae", mae)
    mlflow.log_metric("test_r2_score", r2)
    mlflow.log_metric("training_duration_sec", train_duration)
    
    print("\n--- MODEL PERFORMANCE RESULTS (SEPTEMBER HOLDOUT) ---")
    print(f"  Root Mean Squared Error (RMSE) : {rmse:.2f} rides/hour")
    print(f"  Mean Absolute Error (MAE)      : {mae:.2f} rides/hour")
    print(f"  R-Squared (R²)                 : {r2:.4f}")
    print(f"  Training Duration              : {train_duration:.2f}s")
    
    # Log artifacts to MLflow with explicit dfs_tmpdir
    sample_input = train_df.select(categorical_cols + numerical_cols).limit(5).toPandas()
    sample_pred  = predictions.select("prediction").limit(5).toPandas()
    signature    = infer_signature(sample_input, sample_pred)
    
    mlflow.spark.log_model(
        spark_model=model,
        artifact_path="demand_forecaster_pipeline",
        signature=signature,
        dfs_tmpdir=UC_VOLUME_TMP
    )
    print(f"\nModel logged successfully with Run ID: {run_id}")

In [0]:
print(f"\nRegistering model to Unity Catalog: {MODEL_NAME}...")

model_uri = f"runs:/{run_id}/demand_forecaster_pipeline"
registered_model = mlflow.register_model(
    model_uri=model_uri,
    name=MODEL_NAME
)

print(f"Model successfully registered as Version {registered_model.version} in Unity Catalog.")

In [0]:
print("\nEvaluation Sample: Actual vs. Predicted Demand (Berlin Holdout):")

eval_sample = (
    predictions.filter(F.col("city") == "Berlin")
    .select(
        "city",
        "trip_date",
        "pickup_hour",
        "day_type",
        "weather_condition",
        F.col("trip_count").alias("actual_demand"),
        F.col("prediction").alias("forecasted_demand"),
        F.round(F.abs(F.col("trip_count") - F.col("prediction")), 1).alias("error")
    )
    .orderBy("trip_date", "pickup_hour")
    .limit(10)
)

eval_sample.show(truncate=False)